# Demo: End-to-End GNN–BERT Music Context Inference



In [23]:
import os, glob, sys, subprocess, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def find_input_file(filename, path_hint=None, required=True):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if path_hint:
        hinted = [p for p in matches if path_hint.lower() in p.lower()]
        if hinted:
            matches = hinted
    if not matches:
        if required:
            raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")
        return None
    return matches[0]

TEST_CSV = find_input_file("FINAL_task3_test.csv")
TEST_GRAPH_PATH = find_input_file("FINAL_mtt_test_graphs.pt")
BEST_BERT_PATH = find_input_file("task3_bert_only_best.pt")
GNN_CHECKPOINT_PATH = find_input_file("task3_gnn_only_best.pt")
DISTILBERT_CONFIG = find_input_file("config.json", "distilbert-offline")
DISTILBERT_DIR = os.path.dirname(DISTILBERT_CONFIG)
EARLY_CONCAT_PATH = find_input_file("task3_early_concat_best.pt", required=False)
EARLY_RESULT_JSON = find_input_file("task3_early_concat_results.json", required=False)
PYG_WHEEL = find_input_file("torch_geometric-2.8.0.post1-py3-none-any.whl", "pyg-offline", required=False)

print("Test CSV:", TEST_CSV)
print("Test graphs:", TEST_GRAPH_PATH)
print("BERT checkpoint:", BEST_BERT_PATH)
print("GNN checkpoint:", GNN_CHECKPOINT_PATH)
print("DistilBERT folder:", DISTILBERT_DIR)
print("Fusion checkpoint:", EARLY_CONCAT_PATH)


Device: cuda
Test CSV: /kaggle/input/datasets/jhonsnow07/task4-required-files/FINAL_task3_test.csv
Test graphs: /kaggle/input/datasets/jhonsnow07/task4-required-files/FINAL_mtt_test_graphs.pt
BERT checkpoint: /kaggle/input/datasets/jhonsnow07/task4-required-files/task3_bert_only_best.pt
GNN checkpoint: /kaggle/input/datasets/jhonsnow07/task4-required-files/task3_gnn_only_best.pt
DistilBERT folder: /kaggle/input/datasets/jhonsnow07/distilbert-offline
Fusion checkpoint: /kaggle/input/datasets/jhonsnow07/task3-early-concat-checkpoint/task3_early_concat_best.pt


In [24]:
try:
    import torch_geometric
except ImportError:
    if PYG_WHEEL is None:
        raise FileNotFoundError("Attach pyg-offline because torch_geometric is unavailable.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", PYG_WHEEL])
    import torch_geometric

from torch_geometric.nn import SAGEConv, global_mean_pool
print("PyTorch Geometric:", torch_geometric.__version__)


PyTorch Geometric: 2.8.0.post1


In [25]:
from transformers import AutoTokenizer, AutoConfig, AutoModel

bert_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_DIR, local_files_only=True, use_fast=True)
bert_config = AutoConfig.from_pretrained(DISTILBERT_DIR, local_files_only=True)

TOP50_TAGS = [
    'guitar','classical','slow','techno','strings','drums','electronic','rock','fast','piano',
    'ambient','beat','violin','vocal','synth','female','indian','opera','male','singing',
    'vocals','no vocals','harpsichord','loud','quiet','flute','woman','male vocal','no vocal',
    'pop','soft','sitar','solo','man','classic','choir','voice','new age','dance','female vocal',
    'male voice','beats','harp','cello','no voice','weird','country','metal','female voice','choral'
]
assert len(TOP50_TAGS) == 50
print("Tokenizer ready. Hidden size:", bert_config.hidden_size)


Tokenizer ready. Hidden size: 768


In [26]:
class Task3BERTOnly(nn.Module):
    def __init__(self, config, num_labels=50, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_config(config)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(config.hidden_size, num_labels)
    def encode(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = output.last_hidden_state
        text_embedding = token_embeddings[:, 0, :]
        return text_embedding, token_embeddings
    def forward(self, input_ids, attention_mask):
        text_embedding, _ = self.encode(input_ids, attention_mask)
        return self.classifier(self.dropout(text_embedding))

class Task3GraphSAGE(nn.Module):
    def __init__(self, input_dim=12, hidden_dim=128, num_labels=50, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_labels)
    def encode(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        return global_mean_pool(x, batch)
    def forward(self, x, edge_index, batch):
        return self.classifier(self.encode(x, edge_index, batch))

class EarlyConcatClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(896), nn.Linear(896, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 50)
        )
    def forward(self, x):
        return self.network(x)


In [27]:
bert_model = Task3BERTOnly(bert_config).to(device)
bert_checkpoint = torch.load(BEST_BERT_PATH, map_location=device, weights_only=False)
bert_model.load_state_dict(bert_checkpoint["model_state_dict"], strict=True)
bert_model.eval()

gnn_model = Task3GraphSAGE().to(device)
gnn_checkpoint = torch.load(GNN_CHECKPOINT_PATH, map_location=device, weights_only=False)
gnn_model.load_state_dict(gnn_checkpoint["model_state_dict"], strict=True)
gnn_model.eval()

if EARLY_CONCAT_PATH is None:
    raise FileNotFoundError(
        "Missing task3_early_concat_best.pt. Attach a Kaggle dataset containing the Early-Concat checkpoint generated by Task3_FINAL_CLEAN.ipynb."
    )

concat_model = EarlyConcatClassifier().to(device)
concat_checkpoint = torch.load(EARLY_CONCAT_PATH, map_location=device, weights_only=False)
concat_model.load_state_dict(concat_checkpoint["model_state_dict"], strict=True)
concat_model.eval()

DEMO_THRESHOLD = 0.77
if EARLY_RESULT_JSON is not None:
    with open(EARLY_RESULT_JSON, "r") as f:
        saved_results = json.load(f)
    DEMO_THRESHOLD = float(saved_results.get("best_threshold", DEMO_THRESHOLD))

print("Models restored successfully.")
print("BERT epoch:", bert_checkpoint.get("epoch", "n/a"))
print("GNN epoch:", gnn_checkpoint.get("epoch", "n/a"))
print("Fusion epoch:", concat_checkpoint.get("epoch", "n/a"))
print("Inference threshold:", DEMO_THRESHOLD)


Models restored successfully.
BERT epoch: 4
GNN epoch: 28
Fusion epoch: 11
Inference threshold: 0.77


In [28]:
TEST_DF = pd.read_csv(TEST_CSV)
TEST_GRAPHS = torch.load(TEST_GRAPH_PATH, map_location="cpu", weights_only=False)
assert len(TEST_DF) == len(TEST_GRAPHS)

DEMO_INDEX = None
for i, row_i in TEST_DF.iterrows():
    text_i = str(row_i.get("bert_text", ""))
    has_text = bool(text_i) and "no auxiliary tags" not in text_i.lower()
    has_target = int(row_i[TOP50_TAGS].sum()) > 0
    if has_text and has_target:
        DEMO_INDEX = int(i)
        break
if DEMO_INDEX is None:
    DEMO_INDEX = 0

row = TEST_DF.iloc[DEMO_INDEX]
graph = TEST_GRAPHS[DEMO_INDEX]
print("Demo test index:", DEMO_INDEX)
print("Clip ID:", row.get("clip_id", "n/a"))
print("Auxiliary semantic text:", row["bert_text"])
print("Graph nodes:", graph.x.shape[0])
print("Directed graph edges:", graph.edge_index.shape[1])


Demo test index: 1
Clip ID: 6
Auxiliary semantic text: music context: violins, baroque
Graph nodes: 6
Directed graph edges: 10


In [29]:
text = str(row["bert_text"])
enc = bert_tokenizer(text, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
input_ids = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)

x = graph.x.float().to(device)
edge_index = graph.edge_index.long().to(device)
node_batch = torch.zeros(x.size(0), dtype=torch.long, device=device)

with torch.inference_mode():
    graph_embedding = gnn_model.encode(x, edge_index, node_batch)
    text_embedding, _ = bert_model.encode(input_ids, attention_mask)
    fused = torch.cat([graph_embedding, text_embedding], dim=1)
    logits = concat_model(fused)
    probs = torch.sigmoid(logits)[0].cpu().numpy()

true_tags = [tag for tag in TOP50_TAGS if int(row[tag]) == 1]
predicted = [(TOP50_TAGS[i], float(probs[i])) for i in range(50) if probs[i] >= DEMO_THRESHOLD]
predicted = sorted(predicted, key=lambda x: x[1], reverse=True)
top10_idx = np.argsort(probs)[::-1][:10]
top10 = [(TOP50_TAGS[i], float(probs[i])) for i in top10_idx]

print("\n" + "=" * 72)
print("END-TO-END GNN-BERT INFERENCE")
print("=" * 72)
print("Clip ID:", row.get("clip_id", "n/a"))
print("Semantic input:", text)
print("True target tags:", true_tags)
print("\nPredicted tags at threshold %.2f:" % DEMO_THRESHOLD)
if predicted:
    for tag, p in predicted:
        print(f"  {tag:18s} {p:.4f}")
else:
    print("  No tag exceeded the threshold.")
print("\nTop-10 predicted probabilities:")
for tag, p in top10:
    print(f"  {tag:18s} {p:.4f}")
print("\nEmbedding shapes:")
print("  Graph embedding:", tuple(graph_embedding.shape))
print("  BERT embedding :", tuple(text_embedding.shape))
print("  Fused embedding:", tuple(fused.shape))
print("\nDEMO COMPLETE")



END-TO-END GNN-BERT INFERENCE
Clip ID: 6
Semantic input: music context: violins, baroque
True target tags: ['classical', 'strings', 'violin', 'opera', 'classic']

Predicted tags at threshold 0.77:
  classical          0.9949
  violin             0.9915
  strings            0.9875
  classic            0.9496
  flute              0.9207
  cello              0.9190
  harpsichord        0.8079

Top-10 predicted probabilities:
  classical          0.9949
  violin             0.9915
  strings            0.9875
  classic            0.9496
  flute              0.9207
  cello              0.9190
  harpsichord        0.8079
  slow               0.7381
  no vocal           0.5821
  no vocals          0.5517

Embedding shapes:
  Graph embedding: (1, 128)
  BERT embedding : (1, 768)
  Fused embedding: (1, 896)

DEMO COMPLETE
